# SDS Mock Comprehensive Exam 1 - Solution Notebook

This notebook contains executable reference patterns. The detailed reasoning and rubric are in the worked-solutions DOCX.

In [ ]:
from pathlib import Path
import math, hashlib
DATA_DIR = Path('.')
from pyspark.sql import SparkSession, functions as F
spark = SparkSession.builder.appName('SDS-Mock1-Solutions').getOrCreate()
print('Spark version:', spark.version)

## Problem 1a - recognition
For Jaccard/set similarity use **MinHash**. The single-row collision law is `P[h(A)=h(B)] = J(A,B)`. Random-hyperplane LSH is for cosine/angular similarity.

## Problem 1b - classical banding

In [ ]:
def cand_prob(s,r,b):
    return 1 - (1 - s**r)**b

for r in range(1,61):
    if 60 % r == 0:
        b = 60 // r
        hi = cand_prob(0.80,r,b)
        lo = cand_prob(0.30,r,b)
        if hi >= 0.90 and lo <= 0.05:
            print('VALID', r,b,hi,lo)

## Problem 1c - Spark MinHashLSH

In [ ]:
from pyspark.ml.feature import RegexTokenizer, CountVectorizer, MinHashLSH
from pyspark.ml import Pipeline

raw = spark.read.option('header', True).csv(str(DATA_DIR/'mock1_alert_signatures.csv'))
tok = RegexTokenizer(inputCol='tokens', outputCol='terms', pattern='\\s+')
cv = CountVectorizer(inputCol='terms', outputCol='features', binary=True, minDF=1.0)
prep = Pipeline(stages=[tok, cv]).fit(raw)
X = prep.transform(raw).select('signature_id','features').cache()
model = MinHashLSH(inputCol='features', outputCol='hashes', numHashTables=12).fit(X)
pairs = (model.approxSimilarityJoin(X.alias('a'), X.alias('b'), 0.20, distCol='jaccard_dist')
    .select(F.col('datasetA.signature_id').alias('id1'),
            F.col('datasetB.signature_id').alias('id2'), 'jaccard_dist')
    .filter(F.col('id1') < F.col('id2'))
    .withColumn('estimated_jaccard', 1.0-F.col('jaccard_dist'))
    .orderBy(F.desc('estimated_jaccard'),'id1','id2'))
pairs.show(20,truncate=False)

## Problem 2a - CMS sizing

In [ ]:
eps, delta = 0.001, 0.01
W = math.ceil(math.e/eps)
D = math.ceil(math.log(1/delta))
print(W,D)  # 2719, 5

SEEDS=[17,31,47,61,79]
def build_partition(rows):
    sketch=[[0]*W for _ in range(D)]
    for row in rows:
        key=row.endpoint
        for j,seed in enumerate(SEEDS):
            h=hashlib.sha256(f'{seed}|{key}'.encode()).digest()
            idx=int.from_bytes(h[:8],'big') % W
            sketch[j][idx]+=1
    yield sketch
def merge(a,b):
    return [[a[i][j]+b[i][j] for j in range(W)] for i in range(D)]
stream=spark.read.option('header',True).option('inferSchema',True).csv(str(DATA_DIR/'mock1_api_stream.csv'))
merged=stream.select('endpoint').rdd.mapPartitions(build_partition).reduce(merge)
print('sketch rows,cols=',len(merged),len(merged[0]))

## Problem 2b - AMS

In [ ]:
seq=list('ABACABDA')
from collections import Counter
c=Counter(seq)
F2=sum(v*v for v in c.values())
r=sum(1 for x in seq[2:] if x=='A')
X=len(seq)*(2*r-1)
print(c,F2,r,X)

## Problem 2c - DGIM hand result
Final newest-first buckets: `(1,12), (2,10), (4,6)`. Last-8 estimate = `1+2+4/2=5`; exact count = 4.

## Problem 2d - decay

In [ ]:
alpha=2**(-1/30)
print(alpha,100*(alpha**30)+20)

## Problem 3a - weighted topic-sensitive PageRank

In [ ]:
d=0.85; tol=1e-8; max_iter=100
edges=(spark.read.option('header',True).option('inferSchema',True).csv(str(DATA_DIR/'mock1_news_links.csv'))
       .select('src','dst',F.col('count').cast('double').alias('w'))
       .filter((F.col('src')!=F.col('dst')) & (F.col('w')>0))
       .groupBy('src','dst').agg(F.sum('w').alias('w')).cache())
nodes=(edges.select(F.col('src').alias('id')).union(edges.select(F.col('dst').alias('id'))).distinct().cache())
N=nodes.count()
outw=edges.groupBy('src').agg(F.sum('w').alias('outw')).cache()
trans=(edges.join(outw,'src').withColumn('p',F.col('w')/F.col('outw')).select('src','dst','p').cache())
seeds=(spark.read.option('header',True).option('inferSchema',True).csv(str(DATA_DIR/'mock1_topic_seeds.csv'))
       .select(F.col('node').alias('id'),F.col('weight').cast('double').alias('seed_w')))
seed_sum=seeds.agg(F.sum('seed_w')).first()[0]
v=(nodes.join(seeds,'id','left').fillna(0.0,['seed_w']).withColumn('v',F.col('seed_w')/F.lit(seed_sum)).select('id','v').cache())
ranks=nodes.withColumn('rank',F.lit(1.0/N)).cache()
for it in range(max_iter):
    contrib=(trans.join(ranks,trans.src==ranks.id).select(F.col('dst').alias('id'),(F.col('rank')*F.col('p')).alias('c')).groupBy('id').agg(F.sum('c').alias('incoming')))
    dangling=(ranks.join(outw,ranks.id==outw.src,'left_anti').agg(F.sum('rank').alias('D')).first()['D'] or 0.0)
    newr=(v.join(contrib,'id','left').fillna(0.0,['incoming'])
          .withColumn('rank',(1-F.lit(d))*F.col('v')+F.lit(d)*(F.col('incoming')+F.lit(dangling)*F.col('v')))
          .select('id','rank').cache())
    residual=(ranks.alias('a').join(newr.alias('b'),'id').agg(F.sum(F.abs(F.col('a.rank')-F.col('b.rank')))).first()[0])
    ranks.unpersist(); ranks=newr
    if residual<tol: break
mass=ranks.agg(F.sum('rank')).first()[0]
print('iterations',it+1,'residual',residual,'mass',mass)
ranks.orderBy(F.desc('rank')).show(20,truncate=False)

## Problem 3b - critique
The proposed code ignores edge weights, uses uniform rather than topic teleportation, redistributes dangling mass uniformly rather than by `v`, and uses a fixed 5 iterations without convergence evidence.

## Problem 3c - HITS
`a=A^T h`, `h=Aa`; topic PageRank is directly topic-biased prestige, while HITS separates navigation hubs from authoritative destinations.

## Problem 4a - K_{2,t} seeds

In [ ]:
m=(spark.read.option('header',True).csv(str(DATA_DIR/'mock1_user_project.csv')).select('user','project').dropDuplicates().cache())
pairs=(m.alias('a').join(m.alias('b'),F.col('a.project')==F.col('b.project'))
    .filter(F.col('a.user')<F.col('b.user'))
    .groupBy(F.col('a.user').alias('u'),F.col('b.user').alias('v'))
    .agg(F.countDistinct(F.col('a.project')).alias('shared_count'),
         F.sort_array(F.collect_set(F.col('a.project'))).alias('shared_projects'))
    .filter(F.col('shared_count')>=4).orderBy(F.desc('shared_count'),'u','v'))
pairs.show(20,truncate=False)

## Problem 4b - clique percolation
The two 4-cliques share B,C,D (three nodes = k-1), so they are adjacent under 4-clique percolation. B,C,D are overlap nodes. If they share only two nodes, they are not adjacent for k=4.

## Problem 4c - spectral recognition
`L=D-A`; `Ncut(S,T)=cut/vol(S)+cut/vol(T)`. Use sparse edge joins for normalized-adjacency matvecs, deflate the trivial `sqrt(degree)` direction, normalize and monitor residual. The shifted operator `(I+S)/2` keeps eigenvectors but maps eigenvalues to `[0,1]`, avoiding domination by a negative large-magnitude eigenvalue.